# Libraries

In [1]:
import numpy as np
import pandas as pd
from datasets import load_dataset

In [2]:
from jarvis.core.atoms import Atoms
from pymatgen.core.structure import Structure
import jarvis 

from jarvis.db.figshare import get_jid_data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from jarvis.core.atoms import Atoms, get_supercell_dims
from tqdm import tqdm
from ase.constraints import ExpCellFilter
from sklearn.metrics import mean_absolute_error
import time
from jarvis.core.atoms import ase_to_atoms
from ase.optimize.fire import FIRE
from ase.md.nvtberendsen import NVTBerendsen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase import units
from ase.md.nvtberendsen import NVTBerendsen
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution

from alignn.ff.ff import (
    phonons,
    ForceField,
    AlignnAtomwiseCalculator,
    default_path,
)

model_path = default_path()

def general_relaxer(atoms="", calculator="", fmax=0.05, steps=150): # fmax=0.1, steps=500 
    ase_atoms = atoms.ase_converter()
    ase_atoms.calc = calculator
    ase_atoms = ExpCellFilter(ase_atoms)

    dyn = FIRE(ase_atoms)
    dyn.run(fmax=fmax, steps=steps)
    return ase_to_atoms(ase_atoms.atoms)

calc = AlignnAtomwiseCalculator(
        path=model_path,
        force_mult_natoms=True,
        force_multiplier=1,
        stress_wt=0.3,
    )


dir_path /home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/ff/v5.27.2024


/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/ff/ff.py:284: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


In [3]:
# Models in Alignn
from alignn import pretrained

from pymatgen.analysis.structure_matcher import StructureMatcher
matcher = StructureMatcher(stol=0.5, angle_tol=10, ltol=0.3)
from jarvis.core.atoms import Atoms
from pymatgen.core.structure import Structure
import jarvis 

In [4]:
fourbit_models = [
        "unsloth/tinyllama-chat", #X
        "unsloth/mistral-7b-bnb-4bit", #X
        "unsloth/gemma-7b-bnb-4bit", #X
        "unsloth/llama-3-8b-bnb-4bit", #X 
]  # More models at https://huggingface.co/unsloth

# RANDOMNESS SET

In [5]:
# from alignn import pretrained
# pretrained.get_all_models()

import torch, os
import random
import numpy as np
import pandas as pd
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# Alternatively, you can directly set the device
torch.cuda.set_device(0)  # Replace "1" with the index of the GPU you want to use

def set_seed():
    os.environ["WANDB_ANONYMOUS"] = "must"
    random_seed = 42
    random.seed(random_seed)
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)
    try:
        import torch_xla.core.xla_model as xm
        xm.set_rng_state(random_seed)
    except ImportError:
        pass
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(random_seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = str(":4096:8")
    torch.use_deterministic_algorithms(True)

set_seed()

# Apply Relaxation into Generated Materials

### TC_Supercon

In [37]:
def general_relaxer(atoms="", calculator="", fmax=0.1, steps=250): # fmax=0.1, steps=500 
    ase_atoms = atoms.ase_converter()
    ase_atoms.calc = calculator
    ase_atoms = ExpCellFilter(ase_atoms)

    dyn = FIRE(ase_atoms)
    result = dyn.run(fmax=fmax, steps=steps)
    return result, ase_to_atoms(ase_atoms.atoms)

In [ ]:
for idx, name in enumerate(fourbit_models):

    # Read the model 
    fourbit_model = fourbit_models[idx]
    path = f"./0_gen_tc_supercon/{fourbit_model.split('/')[1]}_generated_samples_updated.csv"
    df = pd.read_csv(path)
    print("********", name)
    
    for jdx, nname in df.iterrows():
        try:
            str_pred = Structure.from_str(df["gen_material_cif"][jdx], fmt="cif")
            atoms_pred = jarvis.core.atoms.pmg_to_atoms(str_pred)
            #formula=atoms_pred.composition.reduced_formula #Reduced formula
            result, opt = general_relaxer(atoms=atoms_pred, calculator=calc)

            if not result: 
                opt = atoms_pred

            str_tar = Structure.from_str(df["orj_material_cif"][jdx], fmt="cif")
            atoms_tar = jarvis.core.atoms.pmg_to_atoms(str_tar).pymatgen_converter()
            rms_dist = matcher.get_rms_anonymous(atoms_tar, opt.pymatgen_converter())
            df.loc[jdx, 'rms_dist_relaxed'] = rms_dist[0]

            try:
                out_data_pred = pretrained.get_prediction(
                                model_name="jv_supercon_tc_alignn",
                                atoms=opt,
                            )
                df.loc[jdx, 'out_data_pred_relaxed'] = out_data_pred[0]
            except Exception as e:
                print('out_data_pred')
                df.loc[jdx, 'out_data_pred_relaxed'] = None
        except Exception as e:            
            print(e)
    break     
    df.to_csv(f"./0_gen_tc_supercon/{fourbit_model.split('/')[1]}_generated_samples_relaxed.csv", index=False)
    del(df)

******** unsloth/tinyllama-chat


/tmp/ipykernel_920602/3107153539.py:34: FutureWarning: Import ExpCellFilter from ase.filters
  ase_atoms = ExpCellFilter(ase_atoms)
/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/torch/autograd/graph.py:825: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at ../aten/src/ATen/cuda/CublasHandlePool.cpp:135.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      Step     Time          Energy          fmax
FIRE:    0 14:33:23      -16.350107        0.007780


### optb88vdw_bandgap

In [9]:
for idx, name in enumerate(fourbit_models):

    # Read the model 
    fourbit_model = fourbit_models[idx]
    path = f"./1_gen_optb88vdw_bandgap/{fourbit_model.split('/')[1]}_generated_samples_updated.csv"
    df = pd.read_csv(path)
    print("********", name)
    
    for jdx, nname in df.iterrows():

        if jdx == 300:
            break
        try:
            str_pred = Structure.from_str(df["gen_material_cif"][jdx], fmt="cif")
            atoms_pred = jarvis.core.atoms.pmg_to_atoms(str_pred)
            #formula=atoms_pred.composition.reduced_formula #Reduced formula
            opt = general_relaxer(atoms=atoms_pred, calculator=calc)

            str_tar = Structure.from_str(df["orj_material_cif"][jdx], fmt="cif")
            atoms_tar = jarvis.core.atoms.pmg_to_atoms(str_tar).pymatgen_converter()
            rms_dist = matcher.get_rms_anonymous(atoms_tar, opt.pymatgen_converter())
            df.loc[jdx, 'rms_dist_relaxed'] = rms_dist[0]

            try:
                out_data_pred = pretrained.get_prediction(
                                model_name="jv_optb88vdw_bandgap_alignn",
                                atoms=opt,
                            )
                df.loc[jdx, 'out_data_pred_relaxed'] = out_data_pred[0]
            except Exception as e:
                print('out_data_pred')
                df.loc[jdx, 'out_data_pred_relaxed'] = None
        except Exception as e:            
            print(e)
        
    df.to_csv(f"./1_gen_optb88vdw_bandgap/{fourbit_model.split('/')[1]}_generated_samples_relaxed.csv", index=False)
    del(df)

******** unsloth/tinyllama-chat


/tmp/ipykernel_895931/2492618894.py:34: FutureWarning: Import ExpCellFilter from ase.filters
  ase_atoms = ExpCellFilter(ase_atoms)


      Step     Time          Energy          fmax
FIRE:    0 00:44:07        1.254992       16.462627
FIRE:    1 00:44:08       -6.794080       12.063778
FIRE:    2 00:44:09      -13.653704        5.288316
FIRE:    3 00:44:10      -16.357596        2.435639
FIRE:    4 00:44:11      -17.935069        2.110594
FIRE:    5 00:44:12      -18.273970        1.159887
FIRE:    6 00:44:13      -18.089066        1.442934
FIRE:    7 00:44:14      -18.133966        1.394690
FIRE:    8 00:44:15      -18.224945        1.282805
FIRE:    9 00:44:16      -18.362330        1.362677
FIRE:   10 00:44:17      -18.655593        1.394947
FIRE:   11 00:44:18      -18.871784        1.473803
FIRE:   12 00:44:19      -19.121169        1.545983
FIRE:   13 00:44:20      -19.298505        1.559554
FIRE:   14 00:44:21      -19.637642        2.033765
FIRE:   15 00:44:22      -19.985851        1.505713
FIRE:   16 00:44:23      -20.295427        1.221642
FIRE:   17 00:44:24      -20.540895        0.947407
FIRE:   18 00:

/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/pretrained.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filenam

      Step     Time          Energy          fmax
FIRE:    0 00:46:37      546.758469      150.299237
FIRE:    1 00:46:40      514.118790      137.114618
FIRE:    2 00:46:42      501.075661      200.871101
FIRE:    3 00:46:45      491.124031       54.505976
FIRE:    4 00:46:47      477.535789       49.549325
FIRE:    5 00:46:50      463.692482       75.523834
FIRE:    6 00:46:52      440.486610      719.409732
FIRE:    7 00:46:55      445.252434      420.981178
FIRE:    8 00:46:57      454.093437      137.233977
FIRE:    9 00:47:00      440.375633      362.380950
FIRE:   10 00:47:02      433.772919      330.766046
FIRE:   11 00:47:05      431.926628      106.087491
FIRE:   12 00:47:07      430.540321      117.140352
FIRE:   13 00:47:10      427.521828       70.891722
FIRE:   14 00:47:13      425.624588       78.532991
FIRE:   15 00:47:15      423.109550       68.615089
FIRE:   16 00:47:18      419.730461      100.653556
FIRE:   17 00:47:20      416.484566       82.090778
FIRE:   18 00:

...

/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/pretrained.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filenam

      Step     Time          Energy          fmax
FIRE:    0 12:34:55      224.221554      485.665052
FIRE:    1 12:34:56       76.596947      398.512441
FIRE:    2 12:34:56       12.371144      170.296234
FIRE:    3 12:34:57      -17.697852       35.227962
FIRE:    4 12:34:58      -21.403294       21.481592
FIRE:    5 12:34:59      -22.216056       35.344537
FIRE:    6 12:35:00      -29.066586       18.884360
FIRE:    7 12:35:00      -35.185692       11.570800
FIRE:    8 12:35:01      -39.822030        8.423098
FIRE:    9 12:35:02      -41.564920        8.308372
FIRE:   10 12:35:03      -42.742263        5.439209
FIRE:   11 12:35:03      -43.600968        2.283613
FIRE:   12 12:35:04      -44.497988        3.823856
FIRE:   13 12:35:05      -44.913114        3.265827
FIRE:   14 12:35:06      -44.952220        3.241696
FIRE:   15 12:35:06      -45.028454        3.152191
FIRE:   16 12:35:07      -45.276383        2.986618
FIRE:   17 12:35:08      -45.408236        2.739612
FIRE:   18 12:

/home/jipengsun/.conda/envs/llm/lib/python3.10/site-packages/alignn/pretrained.py:292: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(filenam

      Step     Time          Energy          fmax
FIRE:    0 12:36:50       26.428771        4.846692
FIRE:    1 12:36:51       16.099093        2.370058


: 

### mbj_bandgap

In [ ]:
for idx, name in enumerate(fourbit_models):

    # Read the model 
    fourbit_model = fourbit_models[idx]
    path = f"./2_gen_mbj_bandgap/{fourbit_model.split('/')[1]}_generated_samples_updated.csv"
    df = pd.read_csv(path)
    print("********", name)
    
    for jdx, nname in df.iterrows():

        if jdx == 300:
            break
        try:
            str_pred = Structure.from_str(df["gen_material_cif"][jdx], fmt="cif")
            atoms_pred = jarvis.core.atoms.pmg_to_atoms(str_pred)
            formula=atoms_pred.composition.reduced_formula #Reduced formula
            opt = general_relaxer(atoms=atoms_pred, calculator=calc)

            str_tar = Structure.from_str(df["orj_material_cif"][jdx], fmt="cif")
            atoms_tar = jarvis.core.atoms.pmg_to_atoms(str_tar).pymatgen_converter()
            rms_dist = matcher.get_rms_anonymous(atoms_tar, opt.pymatgen_converter())
            df.loc[jdx, 'rms_dist_relaxed'] = rms_dist[0]

            try:
                out_data_pred = pretrained.get_prediction(
                                model_name="jv_mbj_bandgap_alignn",
                                atoms=opt,
                            )
                df.loc[jdx, 'out_data_pred_relaxed'] = out_data_pred[0]
            except Exception as e:
                print('out_data_pred')
                df.loc[jdx, 'out_data_pred_relaxed'] = None
        except Exception as e:            
            print(e)
        
    df.to_csv(f"./2_gen_mbj_bandgap/{fourbit_model.split('/')[1]}_generated_samples_relaxed.csv", index=False)
    del(df)